Given a task description, retrieve the correct API or tool to invoke.

Tool-augmented AI agents must select from a library of available skills. Accurate retrieval is critical: the wrong tool produces wrong results even with perfect execution.

**Corpus:** [gorilla-llm/APIBench](https://huggingface.co/datasets/gorilla-llm/APIBench) (HuggingFace Transformers subset) — 300 unique API descriptions with matching task instructions.

**Challenge:** Task instructions describe desired behaviour (`classify sentiment`) while API descriptions state what the model does (`fine-tuned on SST-2 for binary classification`). Paraphrase matching is essential.

In [ ]:
import contextlib, json, pathlib
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BENCHMARK = 'skill-search'
ROOT = pathlib.Path().resolve()
for _p in [ROOT, ROOT.parent, ROOT.parent.parent]:
    if (_p / 'results').exists():
        ROOT = _p; break
RESULTS_DIR = ROOT / 'results'

ADAPTERS = ['sqlite', 'lancedb', 'chromadb', 'tantivy', 'qdrant']
ADAPTER_LABELS = {
    'sqlite': 'SQLite FTS5', 'lancedb': 'LanceDB',
    'chromadb': 'ChromaDB', 'tantivy': 'Tantivy', 'qdrant': 'Qdrant'
}
_OUTER_BG = '#f5f3ef'; _PLOT_BG = '#ffffff'; _MUTED = '#6b6b6b'
_SPINE = '#d8d5d0'; _GRID = '#ebebeb'; _LABEL_CLR = '#7a7370'; _INK = '#1a1917'
_ADAPTER_COLORS = {'sqlite': '#bdb9b5', 'lancedb': '#3d8c7a', 'chromadb': '#4b7ebb', 'tantivy': '#d4952a', 'qdrant': '#edc948'}
_FALLBACK = ['#c96442', '#4b7ebb', '#3d8c7a', '#d4952a', '#bdb9b5']
_P50 = '#e8903a'; _P95 = '#7eb8d4'
_TS = 9; _LS = 8; _TIS = 11.5
mpl.rcParams.update({'figure.dpi': 96, 'font.family': 'sans-serif', 'font.size': _TS,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
    'grid.color': _GRID, 'grid.linewidth': 0.7, 'grid.linestyle': '-', 'axes.axisbelow': True})

def _sty(fig, ax):
    fig.patch.set_facecolor(_OUTER_BG); ax.set_facecolor(_PLOT_BG)
    for s in ['left','bottom']: ax.spines[s].set_color(_SPINE); ax.spines[s].set_linewidth(0.7)
    ax.tick_params(axis='both', colors=_MUTED, labelsize=_TS, length=3, width=0.7)
    ax.xaxis.label.set_color(_MUTED); ax.yaxis.label.set_color(_MUTED)

def _bc(s, i): return _ADAPTER_COLORS.get(s.lower(), _FALLBACK[i % len(_FALLBACK)])

rows = []
for f in RESULTS_DIR.glob('**/*.json'):
    with contextlib.suppress(Exception): rows.append(json.loads(f.read_text()))
df = pd.DataFrame(rows) if rows else pd.DataFrame()
bdf = (df[df['benchmark'] == BENCHMARK]
       .sort_values('ndcg_at_10', ascending=False)
       .groupby('store').first()
       .reindex(ADAPTERS))
print(f'Results for {BENCHMARK}: {len(bdf.dropna(subset=["ndcg_at_10"])) if not bdf.empty else 0} adapters')

## Data and Search Overview

### Tool Routing Flow

```mermaid
flowchart LR
    T["Task description\n(e.g. 'classify sentiment\nin product reviews')"] --> R["Skill retrieval\n(query 300 API descriptions)"]
    R --> S["Best-match API\n(e.g. cardiffnlp/twitter-roberta-base-sentiment)"]
    S --> Call["Agent invokes\ncorrect tool\nwith correct args"]
    style T fill:#fff8e1,stroke:#f9a825
    style Call fill:#e8f5e9,stroke:#2e7d32
```

**The semantic gap challenge:** The task description says "classify sentiment" — the API description says "fine-tuned on SST-2 for binary sentiment classification". No shared tokens; only shared *meaning*.


In [ ]:
import json, pathlib
import matplotlib.pyplot as plt
import numpy as np

ROOT = pathlib.Path().resolve()
for _p in [ROOT, ROOT.parent, ROOT.parent.parent]:
    if (_p / 'results').exists(): ROOT = _p; break

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
fig.patch.set_facecolor('#fafafa')
for ax in axes:
    ax.set_facecolor('#fafafa')
    for s in ax.spines.values(): s.set_visible(False)

# Left: nDCG@10 — dense leads
ax = axes[0]
adapters_ord = ['sqlite', 'tantivy', 'lancedb', 'chromadb']
labels_m = {'sqlite': 'SQLite\nFTS5', 'tantivy': 'Tantivy', 'lancedb': 'LanceDB', 'chromadb': 'ChromaDB'}
cols_m = {'sqlite': '#f28e2b', 'tantivy': '#e15759', 'lancedb': '#4e79a7', 'chromadb': '#59a14f'}
scores = {}
for a in adapters_ord:
    d = ROOT / 'results' / 'skill-search' / a
    fs = sorted(d.glob('*.json')) if d.exists() else []
    if fs: scores[a] = json.loads(fs[-1].read_text()).get('ndcg_at_10', 0)
vals = [scores.get(a, 0) for a in adapters_ord]
bars = ax.bar([labels_m[a] for a in adapters_ord], vals, color=[cols_m[a] for a in adapters_ord], width=0.5, zorder=3)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.005, f'{v:.3f}', ha='center', fontsize=9)
ax.set_ylabel('nDCG@10'); ax.set_ylim(0.6, 1.02)
ax.set_title('Dense search leads: semantic gap favours embeddings', fontsize=9)
ax.yaxis.grid(True, linestyle=':', alpha=0.6)

# Right: illustrative semantic gap (BM25 vs dense on exact vs paraphrase queries)
ax2 = axes[1]
query_types = ['Exact-vocab\nquery', 'Paraphrase\nquery']
bm25_scores = [0.92, 0.18]
dense_scores = [0.91, 0.87]
x = np.arange(len(query_types)); w = 0.3
ax2.bar(x - w/2, bm25_scores, w, color='#f28e2b', label='BM25', zorder=3)
ax2.bar(x + w/2, dense_scores, w, color='#4e79a7', label='Dense', zorder=3)
ax2.set_xticks(x); ax2.set_xticklabels(query_types, fontsize=9)
ax2.set_ylabel('Retrieval score (illustrative)', fontsize=9)
ax2.set_title('BM25 fails on paraphrase queries', fontsize=9)
ax2.set_ylim(0, 1.1); ax2.legend(fontsize=9)
ax2.yaxis.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout(); plt.show()


## Background

### What This Benchmark Measures

Given a natural-language task description, retrieve the correct API or tool to invoke. This directly models how tool-augmented AI agents operate: an agent receives a goal and must select the right function from a library of available tools.

**Corpus:** 300 unique HuggingFace Transformers API descriptions from [gorilla-llm/APIBench](https://huggingface.co/datasets/gorilla-llm/APIBench) (HuggingFace Transformers subset). Each description names a model, its task type, its training data, and its expected input/output format.

**Ground truth:** Each API description comes with 1–3 synthetically generated task instructions that should trigger it. The pairing was validated by Gorilla's authors: only the *functionally correct* API counts as relevant.

**Example pair:**
- *Query:* "I want to classify the sentiment of product reviews as positive or negative"
- *Relevant API:* "cardiffnlp/twitter-roberta-base-sentiment — fine-tuned on SST-2 for binary sentiment classification, returns POSITIVE/NEGATIVE labels"
- *BM25 failure:* zero shared tokens between "classify sentiment" and "fine-tuned on SST-2 for binary classification"

### Why Dense Search Wins Decisively

This is the most demanding semantic gap in the benchmark. Task instructions describe desired *behaviour* using everyday language; API descriptions use ML-specific terminology (model names, training datasets, output formats). Dense vector search succeeds because SBERT embeddings encode meaning — "classify sentiment" and "binary sentiment classification" are geometrically close in the 384-dimensional embedding space even without shared vocabulary.

BM25 still achieves respectable scores (>0.73) because some queries do share tokens with API descriptions (model names, task-type labels), but it fails systematically on paraphrase queries where the vocabulary is completely disjoint.

### References

1. Patil, S. G. et al. (2023). *Gorilla: Large language model connected with massive APIs.* [arXiv:2305.15334](https://arxiv.org/abs/2305.15334)
2. [gorilla-llm/APIBench dataset on HuggingFace](https://huggingface.co/datasets/gorilla-llm/APIBench)
3. Reimers, N. & Gurevych, I. (2019). Sentence-BERT. EMNLP 2019. [arXiv:1908.10084](https://arxiv.org/abs/1908.10084)
4. Schick, T. & Schutze, H. (2021). *Exploiting cloze questions for few-shot text classification.* EACL 2021. *(toolformer ancestor)*
5. [Qdrant: vector database documentation](https://qdrant.tech/documentation/)
6. [sentence-transformers: usage and pre-trained models](https://www.sbert.net/docs/pretrained_models.html)


## Results

In [ ]:
cols = ['Adapter', 'nDCG@10', 'R@1', 'R@5', 'R@10', 'MRR@10', 'p50 (ms)']
rows_t = []
for a in ADAPTERS:
    if bdf.empty or a not in bdf.index or pd.isna(bdf.loc[a].get('ndcg_at_10')): continue
    r = bdf.loc[a]
    rows_t.append({'Adapter': ADAPTER_LABELS[a], 'nDCG@10': f"{r.get('ndcg_at_10',0):.3f}",
        'R@1': f"{r.get('recall_at_1',0):.3f}", 'R@5': f"{r.get('recall_at_5',0):.3f}",
        'R@10': f"{r.get('recall_at_10',0):.3f}", 'MRR@10': f"{r.get('mrr_at_10',0):.3f}",
        'p50 (ms)': f"{r.get('latency_p50_ms',0):.2f}"})
if rows_t:
    from IPython.display import display, HTML
    tdf = pd.DataFrame(rows_t, columns=cols)
    display(HTML(tdf.to_html(index=False, classes='results-table', border=0)))
else:
    from IPython.display import display, HTML
    display(HTML('<p><em>No results.</em></p>'))

In [ ]:
valid = [(a, bdf.loc[a,'ndcg_at_10']) for a in ADAPTERS if not bdf.empty and a in bdf.index and not pd.isna(bdf.loc[a,'ndcg_at_10'])]
if valid:
    stores, vals = zip(*valid)
    labels = [ADAPTER_LABELS.get(s,s) for s in stores]
    colors = [_bc(s,i) for i,s in enumerate(stores)]
    fig, ax = plt.subplots(figsize=(5.5, 3.2))
    _sty(fig, ax)
    bars = ax.bar(labels, vals, color=colors, width=0.5, zorder=3)
    ax.set_ylabel('nDCG@10'); ax.set_ylim(0, 1.1)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.015, f'{val:.3f}',
                ha='center', va='bottom', fontsize=_LS, color=_LABEL_CLR)
    ax.set_title('nDCG@10 by adapter', color=_INK, fontsize=_TIS, fontweight='bold', pad=10)
    plt.tight_layout(pad=1.0); plt.show()

In [ ]:
valid_lat = [(a, bdf.loc[a,'latency_p50_ms'], bdf.loc[a,'latency_p95_ms'])
             for a in ADAPTERS if not bdf.empty and a in bdf.index and not pd.isna(bdf.loc[a].get('latency_p50_ms'))]
fig, ax = plt.subplots(figsize=(5.5, 3.2))
_sty(fig, ax)
if valid_lat:
    stores, p50, p95 = zip(*valid_lat)
    labels = [ADAPTER_LABELS.get(s,s) for s in stores]
    x = np.arange(len(labels)); w = 0.3
    ax.bar(x-w/2, p50, w, label='p50', color=_P50, zorder=3)
    ax.bar(x+w/2, p95, w, label='p95', color=_P95, zorder=3)
    ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel('ms')
    ax.legend(fontsize=_LS, framealpha=0, labelcolor=_MUTED, handlelength=1.0)
else:
    ax.text(0.5, 0.5, 'No latency data', ha='center', va='center', color=_MUTED, transform=ax.transAxes)
ax.set_title('Query latency (ms)', color=_INK, fontsize=_TIS, fontweight='bold', pad=10)
plt.tight_layout(pad=1.0); plt.show()

## Analysis

ChromaDB leads at nDCG@10=0.925, followed by LanceDB (0.889). The dense-vector adapters heavily outperform BM25 (sqlite: 0.749, tantivy: 0.736), confirming that skill routing requires semantic understanding: `classify sentiment in product reviews` must match `fine-tuned on SST-2 for binary classification`.

Even BM25's scores are relatively high (>0.73) because the APIBench corpus contains domain-specific terms (model names, task types) that appear in both instructions and descriptions. On a larger, noisier tool library, the gap between keyword and dense retrieval would widen.

## Limitations

- **HuggingFace Transformers only:** APIBench covers one provider; broader tool libraries (web APIs, shell commands) are not represented.
- **One-to-one qrels:** each instruction has exactly one correct API, but real skill routing may have multiple valid tools.
- **English instructions:** multilingual or code-mixed task descriptions are not evaluated.